In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

# Model & processor
model_path = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"

model = LlavaForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_path)

def explainXray(classification_result, image_path):
  image = Image.open(image_path).convert("RGB")

  # Build prompt
  question_text = f"This brain MRI suggests ${classification_result}. What is the finding in the image that supports this diagnosis?"
  prompt = f"USER: <image>\n{question_text} ASSISTANT:"

  # Prepare input
  inputs = processor(
      images=[image],
      text=prompt,
      return_tensors="pt"
  ).to(model.device, torch.float16)

  # Generate output
  with torch.inference_mode():
      out = model.generate(
          **inputs,
          max_new_tokens=256,
          do_sample=False,
          temperature=0.0,
          pad_token_id=processor.tokenizer.eos_token_id
      )

  # Decode cleanly
  response = processor.decode(out[0], skip_special_tokens=True)
  print("\n--- MRI Findings ---\n")
  print(response.strip())

In [ ]:
explainXray('glioma tumor', 'test_image_1.jpg')

In [ ]:
explainXray('meningioma tumor', 'test_image_2.jpg')

In [ ]:
explainXray('no tumor', 'test_image_3.jpg')

In [ ]:
explainXray('pituitary tumor', 'test_image_4.jpg')